In [1]:
# Importing modules
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Setting up the path to include the parent directory
sys.path.append(str(Path.cwd().parent.parent))
from backend.config.settings import PATHS

In [18]:
water_path = PATHS['definitive_notebooks']/ 'water_definitive.parquet'
water_non_merged_pd = pd.read_parquet(water_path)
water_non_merged_pd.head()

,date,storage,id,storage_imputed
0,1988-01-05,0,3,0
1,1988-01-12,0,3,0
2,1988-01-19,0,3,0
3,1988-01-26,0,3,0
4,1988-02-02,0,3,0


In [20]:
reservoirs_path = PATHS['definitive_notebooks']/ 'reservoirs_merged.parquet'
reservoirs_pd = pd.read_parquet(reservoirs_path)
reservoirs_pd.head()

,id,scope,name,capacity,electric_flag,longitude,latitude,basin,riverbed,google,openstreetmap,wikidata,province,autonomous_community,type,crest_elevation,dam_height,report
0,3,guadalquivir,fernandina,247.0,0,38.179646,-3.570224,guadalquivir,rio guarrizas,None,None,None,jaen,andalucia,presa fabrica gravedad (hormigon vibrado),719.55,NaN,https://sig.mapama.gob.es/WebServices/clientew...
1,5,guadalquivir,puebla cazalla,87.0,0,37.129772,-5.243309,guadalquivir,rio corbones,None,None,None,sevilla,andalucia,presa fabrica gravedad (hormigon compactado),218.25,NaN,https://sig.mapama.gob.es/WebServices/clientew...
2,9,cuenca mediterranea andaluza,vinuela,170.0,0,36.860400,-4.164435,cuencas mediterraneas andaluzas,rio guaro,https://www.google.com/search?kgmid=/g/120jr2sh,None,https://www.wikidata.org/wiki/Q5830515,malaga,andalucia,presa materiales sueltos pantalla hormigon,426.00,NaN,https://sig.mapama.gob.es/WebServices/clientew...
3,11,cuenca mediterranea andaluza,rules,111.0,0,36.860510,-3.495430,cuencas mediterraneas andaluzas,rio guadalfeo o rio cadiar,https://www.google.com/search?kgmid=/g/121vx8sj,None,https://www.wikidata.org/wiki/Q5369455,granada,andalucia,presa fabrica arco gravedad,426.00,NaN,https://sig.mapama.gob.es/WebServices/clientew...
4,12,guadiana,burdalo,79.0,0,-1.112043,42.606125,jucar,riu amadorio,None,None,None,caceres,extremadura,None,320.65,NaN,None


In [21]:
reservoirs_pd.isna().sum()

id                        0
scope                     0
name                      0
capacity                  0
electric_flag             0
longitude                 0
latitude                  0
basin                     0
riverbed                  0
google                  277
openstreetmap           345
wikidata                219
province                  0
autonomous_community      0
type                     86
crest_elevation           0
dam_height              235
report                   86
dtype: int64

Merging the dataframes

In [22]:
water_pd = pd.merge(water_non_merged_pd, reservoirs_pd[['id', 'capacity', 'crest_elevation', 'province', 'autonomous_community']], on='id', how='left')
water_pd.head()

,date,storage,id,storage_imputed,capacity,crest_elevation,province,autonomous_community
0,1988-01-05,0,3,0,247.0,719.55,jaen,andalucia
1,1988-01-12,0,3,0,247.0,719.55,jaen,andalucia
2,1988-01-19,0,3,0,247.0,719.55,jaen,andalucia
3,1988-01-26,0,3,0,247.0,719.55,jaen,andalucia
4,1988-02-02,0,3,0,247.0,719.55,jaen,andalucia


Ensuring that we have data for every week:

In [23]:
water_pd.groupby(['id'])['date'].diff().value_counts()

date
7 days    652380
Name: count, dtype: int64

### Adding year and month features (day is not significant as dates are indexed once a week)

In [24]:
water_pd['year'] = pd.to_datetime(water_pd['date']).dt.year
water_pd['month'] = pd.to_datetime(water_pd['date']).dt.month
water_pd
water_pd.head()

,date,storage,id,storage_imputed,capacity,crest_elevation,province,autonomous_community,year,month
0,1988-01-05,0,3,0,247.0,719.55,jaen,andalucia,1988,1
1,1988-01-12,0,3,0,247.0,719.55,jaen,andalucia,1988,1
2,1988-01-19,0,3,0,247.0,719.55,jaen,andalucia,1988,1
3,1988-01-26,0,3,0,247.0,719.55,jaen,andalucia,1988,1
4,1988-02-02,0,3,0,247.0,719.55,jaen,andalucia,1988,2


### Adding lag features

In [25]:
for lag in [1,2,3,4]:
    water_pd[f'storage_last_week_{lag}'] = water_pd.groupby('id')['storage'].shift(lag)
water_pd['storage_last_year'] = water_pd.groupby('id')['storage'].shift(52)
water_pd.head(15)

,date,storage,id,storage_imputed,capacity,crest_elevation,province,autonomous_community,year,month,storage_last_week_1,storage_last_week_2,storage_last_week_3,storage_last_week_4,storage_last_year
0,1988-01-05,0,3,0,247.0,719.55,jaen,andalucia,1988,1,NaN,NaN,NaN,NaN,NaN
1,1988-01-12,0,3,0,247.0,719.55,jaen,andalucia,1988,1,0.0,NaN,NaN,NaN,NaN
2,1988-01-19,0,3,0,247.0,719.55,jaen,andalucia,1988,1,0.0,0.0,NaN,NaN,NaN
3,1988-01-26,0,3,0,247.0,719.55,jaen,andalucia,1988,1,0.0,0.0,0.0,NaN,NaN
4,1988-02-02,0,3,0,247.0,719.55,jaen,andalucia,1988,2,0.0,0.0,0.0,0.0,NaN
5,1988-02-09,0,3,0,247.0,719.55,jaen,andalucia,1988,2,0.0,0.0,0.0,0.0,NaN
6,1988-02-16,0,3,0,247.0,719.55,jaen,andalucia,1988,2,0.0,0.0,0.0,0.0,NaN
7,1988-02-23,0,3,0,247.0,719.55,jaen,andalucia,1988,2,0.0,0.0,0.0,0.0,NaN
8,1988-03-01,0,3,0,247.0,719.55,jaen,andalucia,1988,3,0.0,0.0,0.0,0.0,NaN
9,1988-03-08,0,3,0,247.0,719.55,jaen,andalucia,1988,3,0.0,0.0,0.0,0.0,NaN


In [26]:
water_pd.isna().sum()

date                        0
storage                     0
id                          0
storage_imputed             0
capacity                    0
crest_elevation             0
province                    0
autonomous_community        0
year                        0
month                       0
storage_last_week_1       374
storage_last_week_2       748
storage_last_week_3      1122
storage_last_week_4      1496
storage_last_year       19448
dtype: int64

### Rolling average and standard deviation for last month

In [28]:
water_pd['storage_mean_4w'] = water_pd.groupby('id')['storage'].rolling(4, min_periods=1).mean().reset_index(level=0, drop=True)
water_pd['storage_std_4w'] = water_pd.groupby('id')['storage'].rolling(4, min_periods=1).std().reset_index(level=0, drop=True)

In [29]:
water_pd.head(15)

,date,storage,id,storage_imputed,capacity,crest_elevation,province,autonomous_community,year,month,storage_last_week_1,storage_last_week_2,storage_last_week_3,storage_last_week_4,storage_last_year,storage_mean_4w,storage_std_4w
0,1988-01-05,0,3,0,247.0,719.55,jaen,andalucia,1988,1,NaN,NaN,NaN,NaN,NaN,0.0,NaN
1,1988-01-12,0,3,0,247.0,719.55,jaen,andalucia,1988,1,0.0,NaN,NaN,NaN,NaN,0.0,0.0
2,1988-01-19,0,3,0,247.0,719.55,jaen,andalucia,1988,1,0.0,0.0,NaN,NaN,NaN,0.0,0.0
3,1988-01-26,0,3,0,247.0,719.55,jaen,andalucia,1988,1,0.0,0.0,0.0,NaN,NaN,0.0,0.0
4,1988-02-02,0,3,0,247.0,719.55,jaen,andalucia,1988,2,0.0,0.0,0.0,0.0,NaN,0.0,0.0
5,1988-02-09,0,3,0,247.0,719.55,jaen,andalucia,1988,2,0.0,0.0,0.0,0.0,NaN,0.0,0.0
6,1988-02-16,0,3,0,247.0,719.55,jaen,andalucia,1988,2,0.0,0.0,0.0,0.0,NaN,0.0,0.0
7,1988-02-23,0,3,0,247.0,719.55,jaen,andalucia,1988,2,0.0,0.0,0.0,0.0,NaN,0.0,0.0
8,1988-03-01,0,3,0,247.0,719.55,jaen,andalucia,1988,3,0.0,0.0,0.0,0.0,NaN,0.0,0.0
9,1988-03-08,0,3,0,247.0,719.55,jaen,andalucia,1988,3,0.0,0.0,0.0,0.0,NaN,0.0,0.0


In [12]:
water_engineered_path = PATHS['engineered_data'] / 'water_engineered.parquet'
water_pd_full = pd.read_parquet(water_engineered_path)
water_pd_full.head()

,date,storage,capacity,crest_elevation,province,autonomous_community,id,year,month,day,storage_missing,storage_last_week_1,storage_last_week_2,storage_last_week_3,storage_last_week_4,storage_last_year,storage_mean_4w,storage_std_4w,completeness,week_idx
0,1988-01-05,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,5,0,NaN,NaN,NaN,NaN,NaN,103.0,NaN,1.0,1
1,1988-01-12,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,12,0,103.0,NaN,NaN,NaN,NaN,103.0,0.0,1.0,2
2,1988-01-19,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,19,0,103.0,103.0,NaN,NaN,NaN,103.0,0.0,1.0,3
3,1988-01-26,103.0,103.0,NaN,cordoba,andalucia,1,1988,1,26,0,103.0,103.0,103.0,NaN,NaN,103.0,0.0,1.0,4
4,1988-02-02,103.0,103.0,NaN,cordoba,andalucia,1,1988,2,2,0,103.0,103.0,103.0,103.0,NaN,103.0,0.0,1.0,5


In [31]:
engineered_water_path = PATHS['engineered_data_notebooks'] / 'water_engineered.parquet'
engineered_water_path.parent.mkdir(parents=True, exist_ok=True)
water_pd.to_parquet(engineered_water_path, index=False)